# AI Lab - Practical Question No. 23
## Maze with Multiple Exits (Multi-Goal Breadth-First Search)

- **Syllabus Area**: Uninformed Search Techniques
- **Difficulty**: 🟡 Intermediate
- **Practical Type**: Algorithm Modification
- **Core AI Concept**: Multi-Goal Search Spaces, BFS Level-Order Optimality, Set Membership Testing
- **Python Concepts Tested**: `collections.deque`, Hash Set `set()` O(1) lookup, Predecessor Backtracking
- **Expected Algorithm / Technique**: Multi-Goal Breadth-First Search (Multi-Goal BFS)
- **Recommended Python Libraries**: `collections`, `typing`, `matplotlib`
- **Complexity Expectation**: Time: $O(b^d)$ where $d$ is depth of closest goal; Space: $O(b^d)$
- **Constraints / Conditions**: Multiple goals exist; only one is required
- **Input Requirements**: 2D Maze array, Start coordinate, List of Exit coordinates
- **Expected Output**: Coords of the optimal exit and the path length
- **Sample Input**: Exits: `(5,5), (1,8)`. Start: `(0,0)`
- **Sample Output**: `Closest exit found at (1,8) in 9 steps.`
- **Optional Challenge**: Return paths to ALL exits sorted by distance
- **University Practical Importance**: High

---

### Problem Statement
Modify a standard Breadth-First Search algorithm to solve a maze that has multiple possible exit points (goals). The algorithm must stop and return the path to the first exit it encounters, which is guaranteed to be the closest exit.

### Task Requirements
1. **Goal State Representation**: Goal state is now a list/set of coordinates $G = \{g_1, g_2, \dots, g_k\}$.
2. **Set Membership Goal Testing**: BFS checks if the current node is in the goal set (`current in goal_set`).
3. **Optimal Exit Output**: Return the coordinate of the found exit and the shortest path length.
4. **Optional Challenge**: Continue exploration to return paths to ALL exits sorted by distance in ascending order.

### 1. Theoretical Foundations: Multi-Goal Search Spaces & BFS Optimality

#### Single-Goal vs. Multi-Goal Formulation
In classical single-goal pathfinding, the search terminates when `current == goal_state`.
In multi-goal search spaces, the target is a set of acceptable goal states:
$$G = \{g_1, g_2, \dots, g_k\}$$
The objective is to find a path from $s_{\text{start}}$ to any goal $g^* \in G$ that minimizes total step distance:
$$d(s_{\text{start}}, g^*) = \min_{g \in G} \text{dist}(s_{\text{start}}, g)$$

#### Why BFS Guarantees Finding the Closest Exit
Breadth-First Search explores the search tree level by level using a First-In-First-Out (FIFO) queue:
$$\text{Level } 0 \to \text{Level } 1 \to \text{Level } 2 \dots \to \text{Level } d$$
Because all transitions on a uniform grid have unit step cost ($c=1$), states are popped from the queue in strictly non-decreasing order of depth. The very first node popped from the frontier that satisfies `current in goal_set` is guaranteed to have the minimal depth $d^*$. No shallower exit can exist in the search space.

#### Efficiency: Hash Set vs. List Membership Testing
- **List Scanning**: Checking `coord in exits_list` takes $O(k)$ time, incurring $O(k \cdot V)$ overhead across $V$ node expansions.
- **Hash Set Lookup**: Converting exits to `set(exits)` yields $O(1)$ average membership testing via hash bucket lookup, keeping the total time bounded by $O(V + k)$ or $O(b^d)$.

In [ ]:
import sys
from pathlib import Path

# Ensure repository root is on sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.multi_goal_bfs_maze import (
    SAMPLE_MULTI_EXIT_MAZE_6X10,
    SAMPLE_START_Q23,
    SAMPLE_EXITS_Q23,
    solve_multi_goal_bfs,
    find_closest_exit,
    find_all_exits_sorted,
    benchmark_set_vs_list_lookup,
    compare_multi_goal_bfs_vs_ucs,
    get_multi_goal_bfs_viva_reference,
    visualize_multi_goal_maze,
)

print("✓ Multi-Goal BFS Maze modules imported successfully!")
print(f"Canonical Start : {SAMPLE_START_Q23}")
print(f"Target Exits    : {SAMPLE_EXITS_Q23}")

### 2. Task Requirements 1, 2 & 3: Executing Multi-Goal BFS on Canonical Sample

We execute the algorithm on the canonical 6x10 benchmark maze:
- Start: `(0, 0)`
- Candidate Exits: `(5, 5)` and `(1, 8)`
- Expected Output: `Closest exit found at (1,8) in 9 steps.`

In [ ]:
# Solve for the closest exit (stopping immediately upon encounter)
closest_exit, steps = find_closest_exit(
    SAMPLE_MULTI_EXIT_MAZE_6X10,
    SAMPLE_START_Q23,
    SAMPLE_EXITS_Q23,
)

print(f"Closest exit found at ({closest_exit[0]},{closest_exit[1]}) in {steps} steps.")

# Detailed result
result = solve_multi_goal_bfs(
    SAMPLE_MULTI_EXIT_MAZE_6X10,
    SAMPLE_START_Q23,
    SAMPLE_EXITS_Q23,
    find_all=False,
    record_trace=True,
)

print(f"\nSolution Status   : {'SUCCESS' if result.found else 'FAILED'}")
print(f"Shortest Path     : {' -> '.join(str(c) for c in result.path)}")
print(f"Action Sequence   : {result.actions_str}")
print(f"Nodes Expanded    : {result.nodes_expanded}")
print(f"Nodes Visited     : {result.nodes_visited}")
print(f"Execution Time    : {result.execution_time_ms:.4f} ms")

print("\n--- ASCII Maze Solution ---")
print(result.render_ascii_solution())

### 3. Optional Challenge: Returning Paths to ALL Exits Sorted by Distance

Instead of stopping at the first goal, the solver continues BFS traversal until all reachable exits in the candidate list are reached. The paths are then reconstructed and sorted by path length in ascending order.

In [ ]:
all_exits_result = solve_multi_goal_bfs(
    SAMPLE_MULTI_EXIT_MAZE_6X10,
    SAMPLE_START_Q23,
    SAMPLE_EXITS_Q23,
    find_all=True,
)

print("All Candidate Exits (Ranked by Shortest Distance):")
for rank, exit_info in enumerate(all_exits_result.all_exits_sorted, 1):
    marker = " [CLOSEST EXIT]" if exit_info.exit_coord == all_exits_result.closest_exit else ""
    print(f"  {rank}. Exit {exit_info.exit_coord}: {exit_info.path_length} steps{marker}")
    print(f"     Path : {' -> '.join(str(c) for c in exit_info.path)}")
    print(f"     Moves: {exit_info.actions_str}")

print("\n--- ASCII Solution with All Paths Rendered ---")
print(all_exits_result.render_ascii_solution(show_all_paths=True))

### 4. Viva Voce Topic 1: Testing for Multiple Goals Efficiently

We empirically benchmark set membership testing (`coord in exits_set`) versus linear list scanning (`coord in exits_list`) over 50,000 queries across 1,000 candidate exit coordinates.

In [ ]:
bench = benchmark_set_vs_list_lookup(num_goals=1000, num_queries=50000)

print("Benchmark Results:")
print(f"  Candidate Goals Count : {bench['num_goals']}")
print(f"  Membership Queries    : {bench['num_queries']:,}")
print(f"  List Linear Scan O(k) : {bench['list_time_ms']:.2f} ms")
print(f"  Set Hash Lookup  O(1) : {bench['set_time_ms']:.2f} ms")
print(f"  Empirical Speedup     : {bench['speedup_factor']:.1f}x FASTER")
print(f"\nTheoretical Summary   : {bench['explanation']}")

### 5. Viva Voce Topic 2: Does Uniform Cost Search (UCS) Still Work with Multiple Goals?

- **Answer**: **YES!** UCS works seamlessly with multiple goals.
- **Core Distinction**: While BFS minimizes the number of steps (hops), UCS expands states in order of cumulative cost $g(n)$, guaranteeing that the first goal popped from the min-heap priority queue is the exit with the **lowest cumulative path cost**.
- **Critical Requirement**: In UCS, the goal test **MUST** be performed upon expansion (popping from the min-heap), NOT upon generation.

In [ ]:
comparison = compare_multi_goal_bfs_vs_ucs()

print("Comparative Analysis: BFS vs UCS with Multiple Goals on Variable Terrain")
print(f"Start Coordinate: {comparison['start']}")
print(f"Candidate Exits : {comparison['exits']}")
print("  - Exit A (0,2): 2 steps through Swamp (cost 10 each, total cost = 20)")
print("  - Exit B (1,3): 4 steps through Road  (cost 1 each,  total cost = 4)")
print("\nResults:")
print(f"  Multi-Goal BFS -> Chosen Exit: {comparison['bfs_chosen_exit']} | Steps: {comparison['bfs_steps']} | Cost: {comparison['bfs_cost']}")
print(f"  Multi-Goal UCS -> Chosen Exit: {comparison['ucs_chosen_exit']} | Steps: {comparison['ucs_steps']} | Cost: {comparison['ucs_cost']}")
print(f"\nViva Takeaway:\n{comparison['viva_takeaway']}")

### 6. Graphical Visualization

We generate a 2D visualization of the maze showing walls, start position, all candidate exits, and the reconstructed optimal path.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize the solution inline
fig, ax = plt.subplots(figsize=(10, 6))
grid_matrix = np.array(result.maze)

cmap = plt.cm.colors.ListedColormap(["#F8F9FA", "#212529"])
ax.imshow(grid_matrix, cmap=cmap, origin="upper")

# Minor grid lines
ax.set_xticks(np.arange(-0.5, len(result.maze[0]), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(result.maze), 1), minor=True)
ax.grid(which="minor", color="#CED4DA", linestyle="-", linewidth=1.5)

# Path to closest exit
pr = [p[0] for p in result.path]
pc = [p[1] for p in result.path]
ax.plot(pc, pr, color="#0D6EFD", linestyle="-", linewidth=3.5, label=f"Optimal Path ({result.path_length} steps)", zorder=4)
ax.scatter(pc, pr, color="#0D6EFD", s=60, zorder=5)

# Start coordinate
sr, sc = result.start
ax.scatter(sc, sr, color="#0D6EFD", s=250, marker="o", edgecolors="black", linewidth=2, label=f"Start {result.start}", zorder=6)
ax.text(sc, sr, "S", color="white", weight="bold", fontsize=11, ha="center", va="center", zorder=7)

# Exits
for e in result.exits:
    er, ec = e
    if e == result.closest_exit:
        ax.scatter(ec, er, color="#198754", s=350, marker="*", edgecolors="black", linewidth=2, label=f"Closest Exit {e} ({result.path_length}s)", zorder=8)
    else:
        ax.scatter(ec, er, color="#DC3545", s=200, marker="X", edgecolors="black", linewidth=1.5, label=f"Other Exit {e}", zorder=6)

ax.set_title(f"Multi-Goal BFS: Closest Exit at {result.closest_exit} in {result.path_length} Steps", fontsize=13, weight="bold", pad=12)
ax.set_xlabel("Column Index", fontsize=11)
ax.set_ylabel("Row Index", fontsize=11)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.0), frameon=True)
plt.tight_layout()
plt.show()

### 7. University Viva Voce Examination Guide

Key theoretical questions and model answers for viva examinations.

In [ ]:
viva_data = get_multi_goal_bfs_viva_reference()
for idx, q_item in enumerate(viva_data["questions"], 1):
    print(f"Q{idx}: {q_item['question']}")
    print(f"Ans: {q_item['answer']}")
    print("-" * 70)